### 🔑 Before you start: your personal lab ID

This lab environment is shared: several students use the same Unity Catalog metastore, and a catalog name can only exist once in a metastore. If everyone created a catalog called `retail_catalog`, only the first student could use it — everyone else would get errors such as *User does not have BROWSE on Catalog*.

To keep your work separate, the first code cell in this notebook works out your personal **lab ID** (8 characters, for example `1a2b3c4d`) and names your catalog **`retail_catalog_<lab ID>`**. Your lab ID is the same in every notebook in this course.

- The setup cell also makes your catalog the **current catalog**, so short names such as `security_lab.customers` automatically point to your catalog. Wherever the instructions mention `retail_catalog`, use your own catalog.
- Where a full name or path is needed, use the Python variable `catalog` (for example `f"/Volumes/{catalog}/..."`), or type the catalog name printed by the setup cell.
- If you take a break and your session restarts, or you see *SCHEMA_NOT_FOUND* / *TABLE_OR_VIEW_NOT_FOUND* errors, **run the setup cell again**.

## Exercise 1: Access Control

You are a data engineer at **NorthMart Retail**, a national supermarket chain. Your team stores customer and transaction data in Unity Catalog. Before any analyst can run queries, you must first create the catalog, the schema, and populate a customer table — then grant the `retail-analysts` group the permissions they need.

> ⚠️ Before starting Task 1.2, ensure you have created your `retail-analysts-<lab ID>` group in Databricks and added your own user account to it. See the lab setup instructions.

> 🔑 **Group name in this lab environment:** group names are shared across the whole environment too, so create the group as **`retail-analysts-<lab ID>`** (the setup cell below prints the exact name and stores it in the Python variable `analyst_group`). Wherever this notebook mentions `retail-analysts`, use your own group.

### Task 1.1 — Create the catalog, schema, and customers table

The cell below sets up all the objects needed for this exercise:
- Works out your personal lab ID and creates your catalog `retail_catalog_<lab ID>` to hold all NorthMart data assets (and makes it your current catalog).
- Creates a schema called `security_lab` inside it.
- Creates a managed Delta table `security_lab.customers` and populates it with six sample customer records. These rows will be used throughout the rest of this exercise to demonstrate row filtering and column masking.

| customer_id | customer_name   | email                   | region | total_spend | tier     |
|-------------|-----------------|-------------------------|--------|-------------|----------|
| 1           | Alice Johnson   | alice@northmart.com     | North  | 5200.00     | Standard |
| 2           | Bob Martinez    | bob@northmart.com       | South  | 8100.00     | Premium  |
| 3           | Carol Lee       | carol@northmart.com     | North  | 3400.00     | Standard |
| 4           | David Chen      | david@northmart.com     | East   | 7800.00     | Premium  |
| 5           | Eve Patel       | eve@northmart.com       | West   | 6200.00     | Standard |
| 6           | Frank Kim       | frank@northmart.com     | South  | 4100.00     | Standard |

Run the cell to provision these objects.

In [ ]:
# Lab environment setup: several students share one Unity Catalog metastore, and a
# catalog name can only exist once in a metastore. Every catalog in this course
# therefore gets your personal lab ID as a suffix. The lab ID is calculated from your
# user name and your workspace, so it is the same in every notebook.
# If your session restarts, simply run this cell again.
import hashlib

if "lab_id" not in globals():
    # Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
    # Use the current catalog to reliably find the workspace default catalog,
    # regardless of its naming convention.
    default_catalog = spark.catalog.currentCatalog()

    storage_root = (
        spark.sql(f"DESCRIBE CATALOG EXTENDED `{default_catalog}`")
        .filter("info_name = 'Storage Root'")
        .select("info_value")
        .first()[0]
    )
    current_user = spark.sql("SELECT current_user()").first()[0]
    lab_id = hashlib.sha1(f"{current_user}|{storage_root}".encode("utf-8")).hexdigest()[:8]

print (f"Storage root: {storage_root}")
print (f"Your lab ID : {lab_id}")

catalog = f"retail_catalog_{lab_id}"

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {catalog}
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Catalog for NorthMart Retail data assets'
""")
spark.sql(f"USE CATALOG {catalog}")
print (f"Your catalog: {catalog}  (now your current catalog)")
analyst_group = f"retail-analysts-{lab_id}"
print (f"Your group  : {analyst_group}  (create it in the workspace settings before Task 1.2)")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS security_lab
  COMMENT 'Schema used for the security lab exercises';

CREATE OR REPLACE TABLE security_lab.customers AS
SELECT * FROM VALUES
  (1, 'Alice Johnson', 'alice@northmart.com', 'North', 5200.00, 'Standard'),
  (2, 'Bob Martinez',  'bob@northmart.com',   'South', 8100.00, 'Premium'),
  (3, 'Carol Lee',     'carol@northmart.com', 'North', 3400.00, 'Standard'),
  (4, 'David Chen',    'david@northmart.com', 'East',  7800.00, 'Premium'),
  (5, 'Eve Patel',     'eve@northmart.com',   'West',  6200.00, 'Standard'),
  (6, 'Frank Kim',     'frank@northmart.com', 'South', 4100.00, 'Standard')
  AS t(customer_id, customer_name, email, region, total_spend, tier);

### Task 1.2 — Grant permissions to the retail-analysts group

Your `retail-analysts-<lab ID>` group needs to query the `customers` table. Grant the necessary privileges at all required levels of the Unity Catalog hierarchy:

- `USE CATALOG` on your `retail_catalog_<lab ID>` catalog
- `USE SCHEMA` on `security_lab`
- `SELECT` on `security_lab.customers`

> 🤖 **Genie Code tip:** Ask:
> *"What SQL grants are needed to allow a Databricks group to SELECT from a Unity Catalog table?"*

**Hint:** Use `GRANT <privilege> ON <object type> <object name> TO \`retail-analysts-<lab ID>\``. Type your real catalog and group names (printed by the setup cell) — group names that contain hyphens must be wrapped in backticks.

In [ ]:
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `{analyst_group}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA security_lab TO `{analyst_group}`")
spark.sql(f"GRANT SELECT ON TABLE security_lab.customers TO `{analyst_group}`")

### Task 1.3 — Verify the granted permissions

Verify that the privileges were correctly assigned. Use `SHOW GRANTS` to display all privileges on the `customers` table.

> 🤖 **Genie Code tip:** Ask:
> *"How do I use SHOW GRANTS to verify permissions on a Unity Catalog table?"*

**Expected result:** You should see your `retail-analysts-<lab ID>` group listed with the `SELECT` privilege.

In [ ]:
%sql
SHOW GRANTS ON TABLE security_lab.customers;

## Exercise 2: Row Filtering

NorthMart Retail's regional analysts should only see customer records for their own region. You will implement a **row filter function** that restricts rows based on who is querying the table.

Since you cannot create multiple user accounts, you will write a filter that shows only `North` region rows to everyone *except* for your own user account, which sees all rows. This simulates what a regional analyst would experience.

### Task 2.1 — Create a row filter function

Create a SQL function `security_lab.region_filter` that takes a `region` parameter of type `STRING` and returns `TRUE` if:
- the `region` is `'North'`, **OR**
- the current user is your own account (replace `your@email.com` with your actual email).

This means: unless you are the privileged user, only `North` region rows will be visible.

> 🤖 **Genie Code tip:** Ask:
> *"How do I create a row filter function in Databricks Unity Catalog that uses current_user() to control access?"*

**Hint:** Use `CREATE OR REPLACE FUNCTION ... RETURN <boolean expression using current_user()>`.

In [ ]:
%sql
-- Replace 'your@email.com' with your actual Azure Databricks user email
CREATE OR REPLACE FUNCTION security_lab.region_filter(region STRING)
RETURN region = 'North' OR current_user() = 'your@email.com';

### Task 2.2 — Apply the row filter to the customers table

Attach the `region_filter` function to the `customers` table so that it is automatically applied to every query against that table.

> 🤖 **Genie Code tip:** Ask:
> *"How do I set a row filter on a Unity Catalog table in Azure Databricks?"*

**Hint:** Use `ALTER TABLE ... SET ROW FILTER <function> ON (<column>)`.

In [ ]:
%sql
ALTER TABLE security_lab.customers
SET ROW FILTER security_lab.region_filter ON (region);

### Task 2.3 — Test the row filter

Query the `customers` table. Because your own email is listed as the privileged user in the filter function, you should see **all 6 rows**.

If you had logged in as someone else (not your email), you would only see the 2 `North` region rows.

> 🤖 **Genie Code tip:** Ask:
> *"How can I verify which user is currently running a query in Databricks using SQL?"*

In [ ]:
%sql
-- Verify the current user
SELECT current_user();

-- Query the customers table — as the privileged user you should see all 6 rows
SELECT * FROM security_lab.customers;

➡️ Tip: update the email address in the row filter function you created earlier, and test again. Observe the output when querying the customers table.

### Task 2.4 — Remove the row filter

Now remove the row filter from the `customers` table. After removing it, query the table again to confirm **all 6 rows** are now returned regardless of user context.

> 🤖 **Genie Code tip:** Ask:
> *"How do I remove a row filter from a Unity Catalog table?"*

**Hint:** Use `ALTER TABLE ... DROP ROW FILTER`.

In [ ]:
%sql
ALTER TABLE security_lab.customers DROP ROW FILTER;

SELECT * FROM security_lab.customers;

## Exercise 3: Column Masking

Customer email addresses are PII and must be masked for users who aren't privileged. You will create a **column mask function** that hides the full email address for most users, applying it to the `email` column of the `customers` table.

### Task 3.1 — Create a column mask function

Create a SQL function `security_lab.mask_email` that takes an `email` parameter of type `STRING` and:
- Returns the **full email** if the current user is your own account.
- Returns a **masked value** (e.g. `'***@***.com'`) for all other users.

> 🤖 **Genie Code tip:** Ask:
> *"How do I create a column masking function in Databricks Unity Catalog that uses current_user() to decide whether to return the real or masked value?"*

**Hint:** Use a `CASE WHEN current_user() = '...' THEN email ELSE '***@***.com' END` pattern inside the function body.

In [ ]:
%sql
-- Replace 'your@email.com' with your actual Azure Databricks user email
CREATE OR REPLACE FUNCTION security_lab.mask_email(email STRING)
RETURN CASE
  WHEN current_user() = 'your@email.com' THEN email
  ELSE '***@***.com'
END;

### Task 3.2 — Apply the column mask to the email column

Apply the `mask_email` function as a column mask on the `email` column of the `customers` table.

> 🤖 **Genie Code tip:** Ask:
> *"How do I set a column mask on a Unity Catalog table column in Azure Databricks using SQL?"*

**Hint:** Use `ALTER TABLE ... ALTER COLUMN email SET MASK <function>`.

In [ ]:
%sql
ALTER TABLE security_lab.customers
ALTER COLUMN email SET MASK security_lab.mask_email;

### Task 3.3 — Test the column mask

Query the `customers` table. Since you are the privileged user specified in the mask function, you should see the **real email addresses**.

Any other user querying this table would see `***@***.com` in the `email` column.

In [ ]:
%sql
-- Your own account sees real emails; any other user sees ***@***.com
SELECT customer_id, customer_name, email, region FROM security_lab.customers;

### Task 3.4 — Remove the column mask

Remove the column mask from the `email` column, then query the table again to confirm emails are now fully visible to all users.

> 🤖 **Genie Code tip:** Ask:
> *"How do I drop a column mask from a Unity Catalog table column?"*

**Hint:** Use `ALTER TABLE ... ALTER COLUMN email DROP MASK`.

In [ ]:
%sql
ALTER TABLE security_lab.customers ALTER COLUMN email DROP MASK;

SELECT customer_id, customer_name, email, region FROM security_lab.customers;

## Exercise 4: Azure Key Vault Secrets

NorthMart Retail's loyalty platform requires an API key to connect. Storing this key in plain text inside a notebook is a security risk. In this exercise you retrieve the API key securely from the Azure Key Vault-backed secret scope you created during lab setup.

> ⚠️ Before running the cells in this exercise, ensure you have completed the Key Vault setup steps in the lab instructions (create Key Vault, add secret `loyalty-api-key`, create secret scope `retail-kv-scope`).

### Task 4.1 — List available secrets in the scope

List all secrets available in the `retail-kv-scope` secret scope to confirm that `loyalty-api-key` is accessible.

> 🤖 **Genie Code tip:** Ask:
> *"How do I list the secrets in a Databricks secret scope using dbutils?"*

**Hint:** Use `dbutils.secrets.list("scope-name")`. This returns key names only — never secret values.

In [ ]:
secrets = dbutils.secrets.list("retail-kv-scope")
for s in secrets:
    print(s.key)

### Task 4.2 — Retrieve the API key securely in Python

Retrieve the `loyalty-api-key` secret from the `retail-kv-scope` scope and assign it to a variable. Then print it.

Observe how Azure Databricks **automatically redacts** the value in the notebook output — you will see `[REDACTED]` instead of the actual key.

> 🤖 **Genie Code tip:** Ask:
> *"How do I retrieve a secret from a Databricks secret scope using dbutils.secrets.get and why is the output redacted?"*

In [ ]:
api_key = dbutils.secrets.get(scope="retail-kv-scope", key="loyalty-api-key")
print(api_key)  # Output will show [REDACTED]

### Task 4.3 — Use the secret to simulate a connection string

Construct a connection string that embeds the API key, simulating how it would be used to connect to NorthMart's loyalty platform. Print the connection string and observe that the secret value is still redacted.

> 🤖 **Genie Code tip:** Ask:
> *"How do I safely use a Databricks secret inside an f-string for a connection string without exposing the value?"*

In [ ]:
connection_string = f"https://loyalty.northmart.com/api?key={api_key}"
print(connection_string)  # The api_key portion will appear as [REDACTED]

### Task 4.4 — Retrieve the secret using SQL

Retrieve the same secret using the `secret()` SQL function. This is useful when constructing SQL-based connections.

Note that the output is also redacted in SQL results.

> 🤖 **Genie Code tip:** Ask:
> *"How do I use the secret() function in Databricks SQL to retrieve a secret from a scope?"*

In [ ]:
%sql
SELECT secret('retail-kv-scope', 'loyalty-api-key') AS loyalty_api_key;

## Cleanup

Run the cell below to remove all objects created during this lab.

In [ ]:
# Remove all lab objects
spark.sql("ALTER TABLE security_lab.customers DROP ROW FILTER")
spark.sql("ALTER TABLE security_lab.customers ALTER COLUMN email DROP MASK")
spark.sql("DROP TABLE IF EXISTS security_lab.customers")
spark.sql("DROP FUNCTION IF EXISTS security_lab.region_filter")
spark.sql("DROP FUNCTION IF EXISTS security_lab.mask_email")
spark.sql("DROP SCHEMA IF EXISTS security_lab")
# Switch back to the workspace default catalog, then drop your own lab catalog
spark.sql(f"USE CATALOG `{default_catalog}`")
spark.sql(f"DROP CATALOG IF EXISTS {catalog} CASCADE")
print(f"Removed {catalog}")